# Quick Start

This notebook goes from an installed package to a lazy dataset in a few steps:

- describe the file convention once, with coordinates, dtype, and filename pattern
- open a folder of binary files as one lazy, Dask-backed {class}`xarray.Dataset`
- compute with the usual xarray API
- write a derived result back to the same raw layout

The sample files are generated on the fly, so the notebook runs as is. To try it on your own data, skip the generation step and point `data_directory` at your folder, adjusting the coordinates, dtype, and pattern to match. The shipped conventions are described in [Spec getter protocols](../usage/spec-getter-protocols.md).

In [ ]:
import pathlib
import tempfile

import numpy as np
import xarray as xr

from xarray_binfile.conventions import Layout, StepIndexedFiles
from xarray_binfile.tutorial import DatasetGenerator

temporary_directory = tempfile.TemporaryDirectory()
data_directory = pathlib.Path(temporary_directory.name)

## Describe the convention

{class}`~xarray_binfile.conventions.Layout` holds what every file has in common: the coordinates in on-disk order and the dtype. {class}`~xarray_binfile.conventions.StepIndexedFiles` adds the filename pattern that tells the files apart, here a variable name and an integer step that becomes the `time` coordinate.

In [ ]:
convention = StepIndexedFiles(
    layout=Layout(
        {"x": np.linspace(0, 1, 8), "y": np.linspace(0, 1, 4)},
        dtype=np.float32,
    ),
    pattern="{name}-{step:04d}.bin",  # ux-0000.bin, ux-0001.bin, ...
)
convention

## Generate sample files

{class}`~xarray_binfile.tutorial.DatasetGenerator` builds random data with the shape and dtype the convention declares for each filename, and {meth}`.binary_engine.to_file() <xarray_binfile.write.BinaryEngineDataset.to_file>` writes it to disk following the same convention. The files are raw `float32` values with no header; the convention is what gives them a shape, coordinates, and a name.

In [ ]:
filenames = [f"{name}-{step:04d}.bin" for name in ("ux", "uy") for step in range(3)]
sample = DatasetGenerator(convention.reader)(map(pathlib.Path, filenames))
sample.binary_engine.to_file(convention.writer, data_directory)
sorted(path.name for path in data_directory.iterdir())

## Open the folder lazily

{meth}`convention.open() <xarray_binfile.conventions.Convention.open>` lists the files that match the pattern and hands them to {func}`xarray.open_mfdataset` with `engine="binfile"`. The `chunks` argument keeps the arrays lazy and Dask-backed, so nothing is read from disk yet. The step from each filename becomes the `time` coordinate, and the files are combined along it.

In [ ]:
dataset = convention.open(data_directory, chunks={"time": 1})
dataset

## Compute with xarray

The object returned above is a regular {class}`xarray.Dataset`, and each variable a regular {class}`xarray.DataArray`. Everything xarray offers applies: indexing by label with {meth}`~xarray.Dataset.sel`, reshaping with {meth}`~xarray.DataArray.expand_dims`, {meth}`~xarray.Dataset.to_dataarray` and {meth}`~xarray.DataArray.to_dataset`, combining with {func}`xarray.merge`, plotting, and so on. Only the chunks a calculation needs are read, and {meth}`~xarray.DataArray.compute` materializes the result in memory.

In [ ]:
mean = dataset["ux"].mean(dim="time").compute()
mean

## Write a derived result back

The same convention writes as well as it reads. The writer takes the variable name from the array and the step from each value along `time`, so the result must keep the dimensions declared in the layout. Here the fluctuation around the time mean is still lazy; writing it computes one file at a time.

In [ ]:
fluctuation = (dataset["ux"] - mean).rename("ux_fluctuation")
fluctuation.binary_engine.to_file(convention.writer, data_directory)
sorted(path.name for path in data_directory.iterdir())

## The standard xarray entry points

{meth}`convention.open() <xarray_binfile.conventions.Convention.open>` is a convenience. The same files open through {func}`xarray.open_dataset` and {func}`xarray.open_mfdataset` with `engine="binfile"`, which fits code that already uses them.

In [ ]:
xr.open_dataset(
    data_directory / "ux-0001.bin",
    engine="binfile",
    read_specs_getter=convention.reader,
)

In [ ]:
temporary_directory.cleanup()

The [read](../tutorials/read.ipynb) and [write](../tutorials/write.ipynb) tutorials expand on each step, and [Parallel and larger-than-memory](../tutorials/parallel-and-larger-than-memory.ipynb) covers Dask schedulers and datasets that do not fit in memory.